In [1]:
from pathlib import Path
import re
import numpy as np

BASE_DIR = Path.cwd()

if BASE_DIR.name == "scripts":
    BASE_DIR = BASE_DIR.parent

GU3_DIR = BASE_DIR / "data" / "gu3"

print("Current working directory:", Path.cwd())
print("BASE_DIR:", BASE_DIR)
print("GU3_DIR:", GU3_DIR)
print("GU3 exists?", GU3_DIR.exists())

Current working directory: f:\code\R16_EEG_analysis
BASE_DIR: f:\code\R16_EEG_analysis
GU3_DIR: f:\code\R16_EEG_analysis\data\gu3
GU3 exists? True


In [2]:
conn_files = sorted(GU3_DIR.glob("*.conn"))

print("Number of .conn files found:", len(conn_files))

for f in conn_files[:5]:
    print(f.name)

Number of .conn files found: 81
M01y09_info1_250Hz_gu3_AEP.conn
NIH01C01_info1_250Hz_gu3_AEP.conn
NIH01C02_info1_250Hz_gu3_AEP.conn
R1629_info1_250Hz_gu3_AEP.conn
R16A01_info1_250Hz_gu2_AEP.conn


In [3]:
conn_path = conn_files[0]
print("Using file:", conn_path.name)

text = conn_path.read_text(encoding="utf-8", errors="ignore")

print(text[:500])

Using file: M01y09_info1_250Hz_gu3_AEP.conn
VersionNumber=1.0 DataType=Coherence DecompositionType=Complex Demodulation ConditionName=gu3 NumberTrials=668 NumberTimeSamples=31 TimeStartInMS=-500.0000 IntervalInMS=50.0000 NumberFrequencies=39 FreqStartInHz=2.0000 FreqIntervalInHz=1.0000 NumberChannels=15 
ACtL ACrL ACaL ACtR ACrR ACaR FL FR PL PR FpM FM CM PM OpM 
1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0


In [4]:
def parse_conn_header(text):
    header_patterns = {
        "NumberTimeSamples": r"NumberTimeSamples=(\d+)",
        "NumberFrequencies": r"NumberFrequencies=(\d+)",
        "FreqStartInHz": r"FreqStartInHz=([0-9.]+)",
        "FreqIntervalInHz": r"FreqIntervalInHz=([0-9.]+)",
        "NumberChannels": r"NumberChannels=(\d+)",
    }

    header = {}

    for key, pattern in header_patterns.items():
        match = re.search(pattern, text)
        if match:
            value = match.group(1)
            header[key] = float(value) if "." in value else int(value)
        else:
            header[key] = None

    lines = text.splitlines()
    channels = None

    for i, line in enumerate(lines):
        if "NumberChannels=" in line:
            if i + 1 < len(lines):
                channels = lines[i + 1].strip().split()
            break

    header["channels"] = channels
    return header


header = parse_conn_header(text)

header

{'NumberTimeSamples': 31,
 'NumberFrequencies': 39,
 'FreqStartInHz': 2.0,
 'FreqIntervalInHz': 1.0,
 'NumberChannels': 15,
 'channels': ['ACtL',
  'ACrL',
  'ACaL',
  'ACtR',
  'ACrR',
  'ACaR',
  'FL',
  'FR',
  'PL',
  'PR',
  'FpM',
  'FM',
  'CM',
  'PM',
  'OpM']}

In [5]:
def extract_numeric_data_after_channels(text):
    lines = text.splitlines()

    data_start_idx = None

    for i, line in enumerate(lines):
        if "NumberChannels=" in line:
            data_start_idx = i + 2
            break

    if data_start_idx is None:
        raise ValueError("Could not find NumberChannels line.")

    data_text = "\n".join(lines[data_start_idx:])

    numbers = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", data_text)
    values = np.array(numbers, dtype=float)

    return values


values = extract_numeric_data_after_channels(text)

expected_4d = (
    header["NumberTimeSamples"]
    * header["NumberFrequencies"]
    * header["NumberChannels"]
    * header["NumberChannels"]
)

expected_3d = (
    header["NumberFrequencies"]
    * header["NumberChannels"]
    * header["NumberChannels"]
)

print("Total numeric values:", len(values))
print("Expected 4D:", expected_4d)
print("Expected 3D:", expected_3d)
print("Match 4D?", len(values) == expected_4d)
print("Match 3D?", len(values) == expected_3d)

Total numeric values: 272025
Expected 4D: 272025
Expected 3D: 8775
Match 4D? True
Match 3D? False


In [6]:
n_time = header["NumberTimeSamples"]
n_freq = header["NumberFrequencies"]
n_ch = header["NumberChannels"]

if len(values) == expected_4d:
    data_4d = values.reshape(n_time, n_freq, n_ch, n_ch)
    print("Data shape:", data_4d.shape)

elif len(values) == expected_3d:
    data_4d = values.reshape(1, n_freq, n_ch, n_ch)
    print("Data had no time dimension. Converted shape:", data_4d.shape)

else:
    raise ValueError("Value count does not match expected 4D or 3D structure.")

Data shape: (31, 39, 15, 15)


In [7]:
data_freq_ch_ch = data_4d.mean(axis=0)

print("After averaging time:", data_freq_ch_ch.shape)

After averaging time: (39, 15, 15)


In [8]:
freqs = header["FreqStartInHz"] + np.arange(header["NumberFrequencies"]) * header["FreqIntervalInHz"]

bands = {
    "delta": (2, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 25),
    "highbeta": (25, 30),
    "gamma": (30, 40),
}

band_matrices = {}

for band_name, (low, high) in bands.items():
    idx = np.where((freqs >= low) & (freqs <= high))[0]
    band_matrices[band_name] = data_freq_ch_ch[idx].mean(axis=0)
    print(band_name, freqs[idx], band_matrices[band_name].shape)

delta [2. 3. 4.] (15, 15)
theta [4. 5. 6. 7. 8.] (15, 15)
alpha [ 8.  9. 10. 11. 12.] (15, 15)
beta [12. 13. 14. 15. 16. 17. 18. 19. 20. 21. 22. 23. 24. 25.] (15, 15)
highbeta [25. 26. 27. 28. 29. 30.] (15, 15)
gamma [30. 31. 32. 33. 34. 35. 36. 37. 38. 39. 40.] (15, 15)


In [9]:
def extract_upper_triangle_features(band_matrices, channels):
    feature_dict = {}

    n_ch = len(channels)
    triu_idx = np.triu_indices(n_ch, k=1)

    for band_name, matrix in band_matrices.items():
        for i, j in zip(*triu_idx):
            ch1 = channels[i]
            ch2 = channels[j]
            feature_name = f"{band_name}_{ch1}_{ch2}"
            feature_dict[feature_name] = matrix[i, j]

    return feature_dict


features = extract_upper_triangle_features(band_matrices, header["channels"])

print("Number of features:", len(features))
list(features.items())[:10]

Number of features: 630


[('delta_ACtL_ACrL', np.float64(0.3228440860215054)),
 ('delta_ACtL_ACaL', np.float64(0.321041935483871)),
 ('delta_ACtL_ACtR', np.float64(0.3214344086021505)),
 ('delta_ACtL_ACrR', np.float64(0.32391182795698925)),
 ('delta_ACtL_ACaR', np.float64(0.3251451612903226)),
 ('delta_ACtL_FL', np.float64(0.3251612903225806)),
 ('delta_ACtL_FR', np.float64(0.3214043010752688)),
 ('delta_ACtL_PL', np.float64(0.31625483870967736)),
 ('delta_ACtL_PR', np.float64(0.3113559139784946)),
 ('delta_ACtL_FpM', np.float64(0.3125688172043011))]

In [ ]:
import pandas as pd

def extract_participant_id(filename):
    """
    Extract the participant ID from the filename.
    For example:
    R16BF02_info1_250Hz_gu3_AEP.conn -> R16BF02
    M01y09_info1_250Hz_gu3_AEP.conn -> M01y09
    """
    return filename.split("_")[0]


def read_conn_file_to_features(conn_path):
    text = conn_path.read_text(encoding="utf-8", errors="ignore")

    header = parse_conn_header(text)
    values = extract_numeric_data_after_channels(text)

    n_time = header["NumberTimeSamples"]
    n_freq = header["NumberFrequencies"]
    n_ch = header["NumberChannels"]

    expected_4d = n_time * n_freq * n_ch * n_ch
    expected_3d = n_freq * n_ch * n_ch

    if len(values) == expected_4d:
        data_4d = values.reshape(n_time, n_freq, n_ch, n_ch)
    elif len(values) == expected_3d:
        data_4d = values.reshape(1, n_freq, n_ch, n_ch)
    else:
        raise ValueError(
            f"{conn_path.name}: value count mismatch. "
            f"Got {len(values)}, expected {expected_4d} or {expected_3d}"
        )

    # Average time dimension
    data_freq_ch_ch = data_4d.mean(axis=0)

    # Frequency list
    freqs = header["FreqStartInHz"] + np.arange(header["NumberFrequencies"]) * header["FreqIntervalInHz"]

    bands = {
        "delta": (2, 4),
        "theta": (4, 8),
        "alpha": (8, 12),
        "beta": (12, 25),
        "highbeta": (25, 30),
        "gamma": (30, 40),
    }

    band_matrices = {}

    for band_name, (low, high) in bands.items():
        idx = np.where((freqs >= low) & (freqs <= high))[0]
        band_matrices[band_name] = data_freq_ch_ch[idx].mean(axis=0)

    features = extract_upper_triangle_features(band_matrices, header["channels"])

    row = {
        "participant_id": extract_participant_id(conn_path.name),
        "filename": conn_path.name,
    }
    row.update(features)

    return row


rows = []
failed_files = []

for conn_path in conn_files:
    try:
        row = read_conn_file_to_features(conn_path)
        rows.append(row)
    except Exception as e:
        failed_files.append((conn_path.name, str(e)))

X_conn_gu3 = pd.DataFrame(rows)

print("Successfully processed files:", len(rows))
print("Failed files:", len(failed_files))
print("Feature table shape:", X_conn_gu3.shape)

if failed_files:
    print("\nFailed file list:")
    for name, err in failed_files:
        print(name, "->", err)

X_conn_gu3.head()

Successfully processed files: 81
Failed files: 0
Feature table shape: (81, 632)


,participant_id,filename,delta_ACtL_ACrL,delta_ACtL_ACaL,delta_ACtL_ACtR,delta_ACtL_ACrR,delta_ACtL_ACaR,delta_ACtL_FL,delta_ACtL_FR,delta_ACtL_PL,...,gamma_FpM_FM,gamma_FpM_CM,gamma_FpM_PM,gamma_FpM_OpM,gamma_FM_CM,gamma_FM_PM,gamma_FM_OpM,gamma_CM_PM,gamma_CM_OpM,gamma_PM_OpM
0,M01y09,M01y09_info1_250Hz_gu3_AEP.conn,0.322844,0.321042,0.321434,0.323912,0.325145,0.325161,0.321404,0.316255,...,0.292410,0.288982,0.288120,0.286998,0.294764,0.295890,0.296091,0.289546,0.287797,0.294152
1,NIH01C01,NIH01C01_info1_250Hz_gu3_AEP.conn,0.327745,0.354146,0.364330,0.358303,0.351795,0.338474,0.326204,0.313004,...,0.328591,0.328390,0.329250,0.330693,0.333187,0.336274,0.333461,0.334664,0.336796,0.335272
2,NIH01C02,NIH01C02_info1_250Hz_gu3_AEP.conn,0.268952,0.269200,0.270895,0.272718,0.270460,0.266839,0.265119,0.266683,...,0.259039,0.259830,0.261043,0.261540,0.264252,0.264313,0.264158,0.262143,0.261891,0.263630
3,R1629,R1629_info1_250Hz_gu3_AEP.conn,0.233116,0.235170,0.235469,0.237892,0.240400,0.240889,0.239189,0.239946,...,0.258612,0.258538,0.258629,0.259438,0.261688,0.261176,0.262242,0.261585,0.261275,0.264169
4,R16A01,R16A01_info1_250Hz_gu2_AEP.conn,0.328522,0.329497,0.337705,0.342174,0.345689,0.342292,0.334226,0.335657,...,0.291259,0.290880,0.291418,0.292474,0.287447,0.287389,0.288823,0.294769,0.294714,0.289187


In [11]:
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

x_output_path = OUTPUT_DIR / "X_conn_gu3.csv"

X_conn_gu3.to_csv(x_output_path, index=False)

print("Saved to:", x_output_path)

Saved to: f:\code\R16_EEG_analysis\outputs\X_conn_gu3.csv


In [12]:
print("Number of unique participants:", X_conn_gu3["participant_id"].nunique())

X_conn_gu3[["participant_id", "filename"]].head(20)

Number of unique participants: 80


,participant_id,filename
0,M01y09,M01y09_info1_250Hz_gu3_AEP.conn
1,NIH01C01,NIH01C01_info1_250Hz_gu3_AEP.conn
2,NIH01C02,NIH01C02_info1_250Hz_gu3_AEP.conn
3,R1629,R1629_info1_250Hz_gu3_AEP.conn
4,R16A01,R16A01_info1_250Hz_gu2_AEP.conn
5,R16A01,R16A01_info1_250Hz_gu3_AEP.conn
6,R16A02g05,R16A02g05_info1_250Hz_gu3_AEP.conn
7,R16A03g07,R16A03g07_info1_250Hz_gu3_AEP.conn
8,R16A04b08,R16A04b08_info1_250Hz_gu3_AEP.conn
9,R16A05y08,R16A05y08_info1_250Hz_gu3_AEP.conn


## Re-run: clean gu3 files only

In [ ]:
# Only keep .conn files whose filenames actually contain _gu3_
conn_files_gu3_only = sorted([f for f in conn_files if "_gu3_" in f.name])

print("Original .conn files:", len(conn_files))
print("True gu3 files:", len(conn_files_gu3_only))

non_gu3_files = [f.name for f in conn_files if "_gu3_" not in f.name]

print("Non-gu3 files found:")
for name in non_gu3_files:
    print(name)

Original .conn files: 81
True gu3 files: 80
Non-gu3 files found:
R16A01_info1_250Hz_gu2_AEP.conn


In [14]:
rows = []
failed_files = []

for conn_path in conn_files_gu3_only:
    try:
        row = read_conn_file_to_features(conn_path)
        rows.append(row)
    except Exception as e:
        failed_files.append((conn_path.name, str(e)))

X_conn_gu3 = pd.DataFrame(rows)

print("Successfully processed gu3 files:", len(rows))
print("Failed files:", len(failed_files))
print("Feature table shape:", X_conn_gu3.shape)
print("Unique participants:", X_conn_gu3["participant_id"].nunique())

if failed_files:
    print("\nFailed file list:")
    for name, err in failed_files:
        print(name, "->", err)

X_conn_gu3[["participant_id", "filename"]].head(20)

Successfully processed gu3 files: 80
Failed files: 0
Feature table shape: (80, 632)
Unique participants: 80


,participant_id,filename
0,M01y09,M01y09_info1_250Hz_gu3_AEP.conn
1,NIH01C01,NIH01C01_info1_250Hz_gu3_AEP.conn
2,NIH01C02,NIH01C02_info1_250Hz_gu3_AEP.conn
3,R1629,R1629_info1_250Hz_gu3_AEP.conn
4,R16A01,R16A01_info1_250Hz_gu3_AEP.conn
5,R16A02g05,R16A02g05_info1_250Hz_gu3_AEP.conn
6,R16A03g07,R16A03g07_info1_250Hz_gu3_AEP.conn
7,R16A04b08,R16A04b08_info1_250Hz_gu3_AEP.conn
8,R16A05y08,R16A05y08_info1_250Hz_gu3_AEP.conn
9,R16BF01,R16BF01_info1_250Hz_gu3_AEP.conn


In [15]:
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

x_output_path = OUTPUT_DIR / "X_conn_gu3.csv"

X_conn_gu3.to_csv(x_output_path, index=False)

print("Saved to:", x_output_path)
print("Shape:", X_conn_gu3.shape)

Saved to: f:\code\R16_EEG_analysis\outputs\X_conn_gu3.csv
Shape: (80, 632)


In [16]:
META_PATH = BASE_DIR / "meta" / "R16 Participant information.xlsx"

print("Excel exists?", META_PATH.exists())
print("Excel path:", META_PATH)

Excel exists? True
Excel path: f:\code\R16_EEG_analysis\meta\R16 Participant information.xlsx


In [17]:
excel_file = pd.ExcelFile(META_PATH)

print("Sheet names:")
print(excel_file.sheet_names)

Sheet names:
['Sheet1']


In [18]:
meta_raw = pd.read_excel(META_PATH)

print("Shape:", meta_raw.shape)
print("Columns:")
print(meta_raw.columns.tolist())

meta_raw.head(10)

Shape: (86, 16)
Columns:
['Participant ID', 'Eprime ID', 'Last Name', 'lang', 'DOB', 'DOE/date of test', 'Toni raw', 'Toni SS', 'PPVT raw', 'PPVT ss', 'TVIP raw', 'TVIP ss', 'Unnamed: 12', 'Language background', 'Music background', 'Notes']


,Participant ID,Eprime ID,Last Name,lang,DOB,DOE/date of test,Toni raw,Toni SS,PPVT raw,PPVT ss,TVIP raw,TVIP ss,Unnamed: 12,Language background,Music background,Notes
0,R16A01y08,NaN,Meduk,S,2014-04-07,2023-03-05,31.0,112.0,204,160,NaN,NaN,NaN,"English, Spanish",yes,NaN
1,R16A02,NaN,Mack,S,2018-10-14,2024-09-18,13.0,102.0,not taken,not taken,NaN,NaN,NaN,"English, Spanish",none,NaN
2,R16A03go7,6031,Hart,E,2017-04-30,2024-10-26,36.0,127.0,172,129,NaN,NaN,NaN,English,none,NaN
3,R16A04,NaN,Belizaire,F,2015-09-23,2024-11-10,22.0,92.0,161,55,NaN,NaN,NaN,"French, Italian, Creole, Spanish, English",none,NaN
4,R16A05er3,17051,Luis,E,2016-07-19,2025-02-07,110.0,75.0,142,93,NaN,NaN,NaN,English,none,NaN
5,R16A06y10,NaN,Guenadiev,S,2017-01-14,2025-10-26,38.0,92.0,163,104,NaN,NaN,NaN,"English, Bulgerian",none,Had voice training
6,R16BF01,1201,Lefebrre,F,2016-01-11,2025-04-06,42.0,127.0,206,160,NaN,NaN,NaN,"English, French",Yes: Piano,NaN
7,R16BF02,1202,Gandhi,F,2016-12-21,2025-04-06,24.0,102.0,177,124,NaN,NaN,NaN,"English, French",Yes: Piano,NaN
8,R16BF04,1204,Blin,S,2016-07-05,2025-04-13,36.0,123.0,156,99,NaN,NaN,NaN,"English, Spanish, French",Yes: Piano and Violin,NaN
9,R16BF05,NaN,Bazabas,S,2019-10-26,2025-04-13,21.0,113.0,125,111,NaN,NaN,NaN,"English, Spanish, French",No,NaN


In [ ]:
metadata_clean = meta_raw.copy()

# Standardize column names
metadata_clean = metadata_clean.rename(columns={
    "Participant ID": "participant_id",
    "DOE/date of test": "date_of_test",
    "Language background": "language_background",
    "Music background": "music_background",
})

# Keep only the columns you currently need.
metadata_clean = metadata_clean[
    [
        "participant_id",
        "lang",
        "DOB",
        "date_of_test",
        "language_background",
        "music_background",
    ]
].copy()

# Clean participant_id and lang
metadata_clean["participant_id"] = metadata_clean["participant_id"].astype(str).str.strip()
metadata_clean["lang"] = metadata_clean["lang"].astype(str).str.strip()

# Date conversion
metadata_clean["DOB"] = pd.to_datetime(metadata_clean["DOB"], errors="coerce")
metadata_clean["date_of_test"] = pd.to_datetime(metadata_clean["date_of_test"], errors="coerce")

# Calculate age at test
metadata_clean["age"] = (
    (metadata_clean["date_of_test"] - metadata_clean["DOB"]).dt.days / 365.25
)

# age group
def make_age_group(age):
    if pd.isna(age):
        return np.nan
    elif 5 <= age <= 7:
        return "5-7"
    elif 8 <= age <= 12:
        return "8-12"
    elif 13 <= age <= 19:
        return "Teens"
    else:
        return "Other"

metadata_clean["age_group"] = metadata_clean["age"].apply(make_age_group)

# Language binary classification：C vs Others
metadata_clean["lang_binary"] = metadata_clean["lang"].apply(
    lambda x: "C" if x == "C" else "Others"
)

metadata_clean.head(10)

,participant_id,lang,DOB,date_of_test,language_background,music_background,age,age_group,lang_binary
0,R16A01y08,S,2014-04-07,2023-03-05,"English, Spanish",yes,8.908966,8-12,Others
1,R16A02,S,2018-10-14,2024-09-18,"English, Spanish",none,5.930185,5-7,Others
2,R16A03go7,E,2017-04-30,2024-10-26,English,none,7.490760,Other,Others
3,R16A04,F,2015-09-23,2024-11-10,"French, Italian, Creole, Spanish, English",none,9.133470,8-12,Others
4,R16A05er3,E,2016-07-19,2025-02-07,English,none,8.555784,8-12,Others
5,R16A06y10,S,2017-01-14,2025-10-26,"English, Bulgerian",none,8.780287,8-12,Others
6,R16BF01,F,2016-01-11,2025-04-06,"English, French",Yes: Piano,9.234771,8-12,Others
7,R16BF02,F,2016-12-21,2025-04-06,"English, French",Yes: Piano,8.290212,8-12,Others
8,R16BF04,S,2016-07-05,2025-04-13,"English, Spanish, French",Yes: Piano and Violin,8.772074,8-12,Others
9,R16BF05,S,2019-10-26,2025-04-13,"English, Spanish, French",No,5.464750,5-7,Others


In [20]:
print("Metadata shape:", metadata_clean.shape)
print("\nLanguage counts:")
print(metadata_clean["lang"].value_counts(dropna=False))

print("\nBinary language counts:")
print(metadata_clean["lang_binary"].value_counts(dropna=False))

print("\nAge group counts:")
print(metadata_clean["age_group"].value_counts(dropna=False))

print("\nMissing DOB:", metadata_clean["DOB"].isna().sum())
print("Missing date_of_test:", metadata_clean["date_of_test"].isna().sum())

metadata_clean[["participant_id", "lang", "lang_binary", "DOB", "date_of_test", "age", "age_group"]].head(20)

Metadata shape: (86, 9)

Language counts:
lang
C      37
S      26
E      12
nan     5
F       3
J       1
K       1
R       1
Name: count, dtype: int64

Binary language counts:
lang_binary
Others    49
C         37
Name: count, dtype: int64

Age group counts:
age_group
8-12     40
5-7      24
Other    11
NaN       6
Teens     5
Name: count, dtype: int64

Missing DOB: 6
Missing date_of_test: 2


,participant_id,lang,lang_binary,DOB,date_of_test,age,age_group
0,R16A01y08,S,Others,2014-04-07,2023-03-05,8.908966,8-12
1,R16A02,S,Others,2018-10-14,2024-09-18,5.930185,5-7
2,R16A03go7,E,Others,2017-04-30,2024-10-26,7.490760,Other
3,R16A04,F,Others,2015-09-23,2024-11-10,9.133470,8-12
4,R16A05er3,E,Others,2016-07-19,2025-02-07,8.555784,8-12
5,R16A06y10,S,Others,2017-01-14,2025-10-26,8.780287,8-12
6,R16BF01,F,Others,2016-01-11,2025-04-06,9.234771,8-12
7,R16BF02,F,Others,2016-12-21,2025-04-06,8.290212,8-12
8,R16BF04,S,Others,2016-07-05,2025-04-13,8.772074,8-12
9,R16BF05,S,Others,2019-10-26,2025-04-13,5.464750,5-7


In [21]:
metadata_output_path = OUTPUT_DIR / "metadata_clean.csv"

metadata_clean.to_csv(metadata_output_path, index=False)

print("Saved to:", metadata_output_path)
print("Shape:", metadata_clean.shape)

Saved to: f:\code\R16_EEG_analysis\outputs\metadata_clean.csv
Shape: (86, 9)


In [ ]:
dataset_gu3 = X_conn_gu3.merge(
    metadata_clean,
    on="participant_id",
    how="left"
)

print("X_conn_gu3 shape:", X_conn_gu3.shape)
print("metadata_clean shape:", metadata_clean.shape)
print("dataset_gu3 shape:", dataset_gu3.shape)

# Check which EEG files do not match metadata
missing_meta = dataset_gu3[dataset_gu3["lang"].isna()][["participant_id", "filename"]]

print("Rows without metadata:", len(missing_meta))
missing_meta.head(20)

X_conn_gu3 shape: (80, 632)
metadata_clean shape: (86, 9)
dataset_gu3 shape: (80, 640)
Rows without metadata: 39


,participant_id,filename
0,M01y09,M01y09_info1_250Hz_gu3_AEP.conn
1,NIH01C01,NIH01C01_info1_250Hz_gu3_AEP.conn
2,NIH01C02,NIH01C02_info1_250Hz_gu3_AEP.conn
3,R1629,R1629_info1_250Hz_gu3_AEP.conn
4,R16A01,R16A01_info1_250Hz_gu3_AEP.conn
5,R16A02g05,R16A02g05_info1_250Hz_gu3_AEP.conn
6,R16A03g07,R16A03g07_info1_250Hz_gu3_AEP.conn
7,R16A04b08,R16A04b08_info1_250Hz_gu3_AEP.conn
8,R16A05y08,R16A05y08_info1_250Hz_gu3_AEP.conn
12,R16BF05y05,R16BF05y05_info1_250Hz_gu3_AEP.conn


In [ ]:
def make_canonical_id(pid):
    """
    Unify the participant IDs in the conn file and Excel with a matching base ID.
    rule：
    - R16A02g05 -> R16A02
    - R16A03g07 -> R16A03
    - R16A04b08 -> R16A04
    - R16C05y13speech -> R16C05
    - R16C06y15b -> R16C06
    - r16c20 -> R16C20
    - R16GTest save as R16GTEST
    """
    if pd.isna(pid):
        return np.nan

    pid = str(pid).strip()
    pid_upper = pid.upper()

    # Matching criteria: R16 + letter + two digits, e.g., R16A02, R16C05, R16E08
    match = re.match(r"^(R16[A-Z]\d{2})", pid_upper)
    if match:
        return match.group(1)

    # Special case: R1629, which has no letters, is retained.
    match = re.match(r"^(R16\d+)", pid_upper)
    if match:
        return match.group(1)

    # Other codes such as M01y09 and NIH01C01 should be kept in uppercase.
    return pid_upper


X_conn_gu3["canonical_id"] = X_conn_gu3["participant_id"].apply(make_canonical_id)
metadata_clean["canonical_id"] = metadata_clean["participant_id"].apply(make_canonical_id)

print("X unique original IDs:", X_conn_gu3["participant_id"].nunique())
print("X unique canonical IDs:", X_conn_gu3["canonical_id"].nunique())

print("Metadata unique original IDs:", metadata_clean["participant_id"].nunique())
print("Metadata unique canonical IDs:", metadata_clean["canonical_id"].nunique())

X_conn_gu3[["participant_id", "canonical_id", "filename"]].head(30)

X unique original IDs: 80
X unique canonical IDs: 79
Metadata unique original IDs: 86
Metadata unique canonical IDs: 84


,participant_id,canonical_id,filename
0,M01y09,M01Y09,M01y09_info1_250Hz_gu3_AEP.conn
1,NIH01C01,NIH01C01,NIH01C01_info1_250Hz_gu3_AEP.conn
2,NIH01C02,NIH01C02,NIH01C02_info1_250Hz_gu3_AEP.conn
3,R1629,R1629,R1629_info1_250Hz_gu3_AEP.conn
4,R16A01,R16A01,R16A01_info1_250Hz_gu3_AEP.conn
5,R16A02g05,R16A02,R16A02g05_info1_250Hz_gu3_AEP.conn
6,R16A03g07,R16A03,R16A03g07_info1_250Hz_gu3_AEP.conn
7,R16A04b08,R16A04,R16A04b08_info1_250Hz_gu3_AEP.conn
8,R16A05y08,R16A05,R16A05y08_info1_250Hz_gu3_AEP.conn
9,R16BF01,R16BF01,R16BF01_info1_250Hz_gu3_AEP.conn


In [24]:
dataset_gu3 = X_conn_gu3.merge(
    metadata_clean,
    on="canonical_id",
    how="left",
    suffixes=("_conn", "_meta")
)

print("X_conn_gu3 shape:", X_conn_gu3.shape)
print("metadata_clean shape:", metadata_clean.shape)
print("dataset_gu3 shape:", dataset_gu3.shape)

missing_meta = dataset_gu3[dataset_gu3["lang"].isna()][
    ["participant_id_conn", "canonical_id", "filename"]
]

print("Rows without metadata:", len(missing_meta))
missing_meta.head(50)

X_conn_gu3 shape: (80, 633)
metadata_clean shape: (86, 10)
dataset_gu3 shape: (81, 642)
Rows without metadata: 14


,participant_id_conn,canonical_id,filename
0,M01y09,M01Y09,M01y09_info1_250Hz_gu3_AEP.conn
1,NIH01C01,NIH01C01,NIH01C01_info1_250Hz_gu3_AEP.conn
2,NIH01C02,NIH01C02,NIH01C02_info1_250Hz_gu3_AEP.conn
3,R1629,R1629,R1629_info1_250Hz_gu3_AEP.conn
12,R16BF05y05,R16BF05Y05,R16BF05y05_info1_250Hz_gu3_AEP.conn
49,R16CjTest,R16CJTEST,R16CjTest_info1_250Hz_gu3_AEP.conn
57,R16jy24,R16JY24,R16jy24_info1_250Hz_gu3_AEP.conn
58,R16L01,R16L01,R16L01_info1_250Hz_gu3_AEP.conn
62,R16me08,R16ME08,R16me08_info1_250Hz_gu3_AEP.conn
63,R16mme07gu,R16MME07GU,R16mme07gu_info1_250Hz_gu3_AEP.conn


In [25]:
matched_dataset = dataset_gu3[dataset_gu3["lang"].notna()].copy()

print("Matched rows:", matched_dataset.shape[0])
print("Unmatched rows:", dataset_gu3["lang"].isna().sum())

print("\nLanguage counts in matched dataset:")
print(matched_dataset["lang"].value_counts(dropna=False))

print("\nBinary language counts in matched dataset:")
print(matched_dataset["lang_binary"].value_counts(dropna=False))

print("\nAge group counts in matched dataset:")
print(matched_dataset["age_group"].value_counts(dropna=False))

matched_dataset[[
    "participant_id_conn",
    "canonical_id",
    "participant_id_meta",
    "filename",
    "lang",
    "lang_binary",
    "age",
    "age_group"
]].head(30)

Matched rows: 67
Unmatched rows: 14

Language counts in matched dataset:
lang
C      30
S      21
E       8
F       3
nan     3
J       1
K       1
Name: count, dtype: int64

Binary language counts in matched dataset:
lang_binary
Others    37
C         30
Name: count, dtype: int64

Age group counts in matched dataset:
age_group
8-12     32
5-7      19
Other    10
Teens     3
NaN       3
Name: count, dtype: int64


,participant_id_conn,canonical_id,participant_id_meta,filename,lang,lang_binary,age,age_group
4,R16A01,R16A01,R16A01y08,R16A01_info1_250Hz_gu3_AEP.conn,S,Others,8.908966,8-12
5,R16A02g05,R16A02,R16A02,R16A02g05_info1_250Hz_gu3_AEP.conn,S,Others,5.930185,5-7
6,R16A03g07,R16A03,R16A03go7,R16A03g07_info1_250Hz_gu3_AEP.conn,E,Others,7.490760,Other
7,R16A04b08,R16A04,R16A04,R16A04b08_info1_250Hz_gu3_AEP.conn,F,Others,9.133470,8-12
8,R16A05y08,R16A05,R16A05er3,R16A05y08_info1_250Hz_gu3_AEP.conn,E,Others,8.555784,8-12
9,R16BF01,R16BF01,R16BF01,R16BF01_info1_250Hz_gu3_AEP.conn,F,Others,9.234771,8-12
10,R16BF02,R16BF02,R16BF02,R16BF02_info1_250Hz_gu3_AEP.conn,F,Others,8.290212,8-12
11,R16BF04,R16BF04,R16BF04,R16BF04_info1_250Hz_gu3_AEP.conn,S,Others,8.772074,8-12
13,R16BF06,R16BF06,R16BF06,R16BF06_info1_250Hz_gu3_AEP.conn,S,Others,8.487337,8-12
14,R16C04,R16C04,R16C04,R16C04_info1_250Hz_gu3_AEP.conn,C,C,13.880903,Teens


In [ ]:
# Re-preparing ID
X_tmp = X_conn_gu3.copy()
M_tmp = metadata_clean.copy()

X_tmp["exact_id"] = X_tmp["participant_id"].astype(str).str.strip().str.upper()
M_tmp["exact_id"] = M_tmp["participant_id"].astype(str).str.strip().str.upper()

X_tmp["canonical_id"] = X_tmp["participant_id"].apply(make_canonical_id)
M_tmp["canonical_id"] = M_tmp["participant_id"].apply(make_canonical_id)

# The metadata is marked as indicating whether this line is the base ID itself.
M_tmp["is_base_id"] = M_tmp["exact_id"] == M_tmp["canonical_id"]

print("Metadata canonical duplicates:")
dup_meta = M_tmp[M_tmp["canonical_id"].duplicated(keep=False)].sort_values("canonical_id")
dup_meta[["participant_id", "exact_id", "canonical_id", "is_base_id", "lang", "age", "age_group"]]

Metadata canonical duplicates:


,participant_id,exact_id,canonical_id,is_base_id,lang,age,age_group
13,R16C05,R16C05,R16C05,True,S,13.180014,Teens
14,R16C05Y16,R16C05Y16,R16C05,False,J,16.678987,Teens
58,R16E06,R16E06,R16E06,True,E,5.095140,5-7
59,R16E06Y08,R16E06Y08,R16E06,False,E,8.654346,8-12


In [ ]:
# 对每个 canonical_id，只保留一条 metadata
# 优先保留 exact_id == canonical_id 的基础 ID row
M_sorted = M_tmp.sort_values(
    by=["canonical_id", "is_base_id"],
    ascending=[True, False]
)

M_unique = M_sorted.drop_duplicates(subset=["canonical_id"], keep="first").copy()

print("Original metadata rows:", M_tmp.shape[0])
print("Unique metadata rows by canonical_id:", M_unique.shape[0])

# 检查是否还有 canonical_id 重复
print("Remaining duplicate canonical IDs:", M_unique["canonical_id"].duplicated().sum())

M_unique[["participant_id", "exact_id", "canonical_id", "is_base_id", "lang", "age", "age_group"]].head(30)

Original metadata rows: 86
Unique metadata rows by canonical_id: 84
Remaining duplicate canonical IDs: 0


,participant_id,exact_id,canonical_id,is_base_id,lang,age,age_group
0,R16A01y08,R16A01Y08,R16A01,False,S,8.908966,8-12
1,R16A02,R16A02,R16A02,True,S,5.930185,5-7
2,R16A03go7,R16A03GO7,R16A03,False,E,7.490760,Other
3,R16A04,R16A04,R16A04,True,F,9.133470,8-12
4,R16A05er3,R16A05ER3,R16A05,False,E,8.555784,8-12
5,R16A06y10,R16A06Y10,R16A06,False,S,8.780287,8-12
6,R16BF01,R16BF01,R16BF01,True,F,9.234771,8-12
7,R16BF02,R16BF02,R16BF02,True,F,8.290212,8-12
8,R16BF04,R16BF04,R16BF04,True,S,8.772074,8-12
9,R16BF05,R16BF05,R16BF05,True,S,5.464750,5-7


In [28]:
dataset_gu3 = X_tmp.merge(
    M_unique.drop(columns=["exact_id"]),
    on="canonical_id",
    how="left",
    suffixes=("_conn", "_meta")
)

print("X_tmp shape:", X_tmp.shape)
print("M_unique shape:", M_unique.shape)
print("dataset_gu3 shape:", dataset_gu3.shape)

missing_meta = dataset_gu3[dataset_gu3["lang"].isna()][
    ["participant_id_conn", "canonical_id", "filename"]
]

print("Rows without metadata:", len(missing_meta))
missing_meta

X_tmp shape: (80, 634)
M_unique shape: (84, 12)
dataset_gu3 shape: (80, 644)
Rows without metadata: 14


,participant_id_conn,canonical_id,filename
0,M01y09,M01Y09,M01y09_info1_250Hz_gu3_AEP.conn
1,NIH01C01,NIH01C01,NIH01C01_info1_250Hz_gu3_AEP.conn
2,NIH01C02,NIH01C02,NIH01C02_info1_250Hz_gu3_AEP.conn
3,R1629,R1629,R1629_info1_250Hz_gu3_AEP.conn
12,R16BF05y05,R16BF05Y05,R16BF05y05_info1_250Hz_gu3_AEP.conn
48,R16CjTest,R16CJTEST,R16CjTest_info1_250Hz_gu3_AEP.conn
56,R16jy24,R16JY24,R16jy24_info1_250Hz_gu3_AEP.conn
57,R16L01,R16L01,R16L01_info1_250Hz_gu3_AEP.conn
61,R16me08,R16ME08,R16me08_info1_250Hz_gu3_AEP.conn
62,R16mme07gu,R16MME07GU,R16mme07gu_info1_250Hz_gu3_AEP.conn


In [ ]:
dataset_lang = dataset_gu3.copy()

# remove the string form of nan.
dataset_lang["lang"] = dataset_lang["lang"].replace(["nan", "NaN", "None", ""], np.nan)

# The language classification only retains lines containing "lang".
dataset_lang = dataset_lang[dataset_lang["lang"].notna()].copy()

# Regenerate binary labels to prevent the string "nan" from being classified as "Others".
dataset_lang["lang_binary"] = dataset_lang["lang"].apply(
    lambda x: "C" if x == "C" else "Others"
)

print("Language dataset shape:", dataset_lang.shape)
print("\nLanguage counts:")
print(dataset_lang["lang"].value_counts(dropna=False))

print("\nBinary language counts:")
print(dataset_lang["lang_binary"].value_counts(dropna=False))

dataset_lang[["participant_id_conn", "canonical_id", "participant_id_meta", "filename", "lang", "lang_binary", "age", "age_group"]].head(30)

Language dataset shape: (63, 644)

Language counts:
lang
C    30
S    21
E     8
F     3
K     1
Name: count, dtype: int64

Binary language counts:
lang_binary
Others    33
C         30
Name: count, dtype: int64


,participant_id_conn,canonical_id,participant_id_meta,filename,lang,lang_binary,age,age_group
4,R16A01,R16A01,R16A01y08,R16A01_info1_250Hz_gu3_AEP.conn,S,Others,8.908966,8-12
5,R16A02g05,R16A02,R16A02,R16A02g05_info1_250Hz_gu3_AEP.conn,S,Others,5.930185,5-7
6,R16A03g07,R16A03,R16A03go7,R16A03g07_info1_250Hz_gu3_AEP.conn,E,Others,7.490760,Other
7,R16A04b08,R16A04,R16A04,R16A04b08_info1_250Hz_gu3_AEP.conn,F,Others,9.133470,8-12
8,R16A05y08,R16A05,R16A05er3,R16A05y08_info1_250Hz_gu3_AEP.conn,E,Others,8.555784,8-12
9,R16BF01,R16BF01,R16BF01,R16BF01_info1_250Hz_gu3_AEP.conn,F,Others,9.234771,8-12
10,R16BF02,R16BF02,R16BF02,R16BF02_info1_250Hz_gu3_AEP.conn,F,Others,8.290212,8-12
11,R16BF04,R16BF04,R16BF04,R16BF04_info1_250Hz_gu3_AEP.conn,S,Others,8.772074,8-12
13,R16BF06,R16BF06,R16BF06,R16BF06_info1_250Hz_gu3_AEP.conn,S,Others,8.487337,8-12
14,R16C04,R16C04,R16C04,R16C04_info1_250Hz_gu3_AEP.conn,C,C,13.880903,Teens


In [30]:
dataset_output_path = OUTPUT_DIR / "dataset_gu3.csv"
dataset_lang_output_path = OUTPUT_DIR / "dataset_lang_gu3.csv"

dataset_gu3.to_csv(dataset_output_path, index=False)
dataset_lang.to_csv(dataset_lang_output_path, index=False)

print("Saved full dataset to:", dataset_output_path)
print("Full dataset shape:", dataset_gu3.shape)

print("Saved language dataset to:", dataset_lang_output_path)
print("Language dataset shape:", dataset_lang.shape)

Saved full dataset to: f:\code\R16_EEG_analysis\outputs\dataset_gu3.csv
Full dataset shape: (80, 644)
Saved language dataset to: f:\code\R16_EEG_analysis\outputs\dataset_lang_gu3.csv
Language dataset shape: (63, 644)
